**Nível 1: Básico — Configuração e SQL Puro com Segurança**

In [6]:
import os
import pandas as pd

import pandas as pd

from sqlalchemy import (
    create_engine,
    text,
    Table,
    MetaData,
    Column,
    Integer,
    String,
    Float,
    ForeignKey,
    insert,
    update,
    select,
    func
)

from sqlalchemy.orm import (
    declarative_base,
    mapped_column,
    relationship,
    sessionmaker,
    Mapped
)


engine = create_engine("sqlite:///sistema_rh.db")

In [7]:
with engine.begin() as conn:
    conn.execute(text("""
        CREATE TABLE IF NOT EXISTS funcionarios (
            id INTEGER PRIMARY KEY AUTOINCREMENT,
            nome VARCHAR(100) NOT NULL,
            cargo VARCHAR(100) NOT NULL,
            salario FLOAT NOT NULL
        )
    """))

In [9]:
nome = "Davi"
cargo = "Desenvolvedor Júnior"
salario = 3500.00

with engine.begin() as conn:
    conn.execute(
        text("""
            INSERT INTO funcionarios (nome, cargo, salario)
            VALUES (:nome, :cargo, :salario)
        """),
        {
            "nome": nome,
            "cargo": cargo,
            "salario": salario
        }
    )

In [10]:
df = pd.read_sql_query(
    "SELECT * FROM funcionarios",
    engine
)

print("Funcionários:")
print(df)

Funcionários:
   id  nome                 cargo  salario
0   1  Davi  Desenvolvedor Júnior   3500.0


**Nível 2: Intermediário — SQLAlchemy Core (Automatização Programática)**

In [11]:
metadata = MetaData()


projetos = Table(
    "projetos",
    metadata,

    Column(
        "id",
        Integer,
        primary_key=True,
        autoincrement=True
    ),

    Column(
        "nome",
        String(100),
        nullable=False
    ),

    Column(
        "descricao",
        String(255)
    ),

    Column(
        "orcamento",
        Float
    )
)


metadata.create_all(engine)


In [12]:
lista_de_dicts = [
    {
        "nome": "Sistema de RH",
        "descricao": "Sistema para gerenciamento de funcionários",
        "orcamento": 45000.00
    },
    {
        "nome": "Aplicativo Mobile",
        "descricao": "Aplicativo interno",
        "orcamento": 25000.00
    },
    {
        "nome": "Dashboard Financeiro",
        "descricao": "Dashboard de indicadores",
        "orcamento": 12000.00
    }
]


with engine.begin() as conn:

    conn.execute(
        insert(projetos),
        lista_de_dicts
    )

In [13]:
funcionarios = Table(
    "funcionarios",
    metadata,
    autoload_with=engine
)

In [14]:
with engine.begin() as conn:

    conn.execute(
        update(funcionarios)
        .where(
            funcionarios.c.cargo == "Desenvolvedor Júnior"
        )
        .values(
            salario=funcionarios.c.salario * 1.10
        )
    )

In [15]:
consulta = (
    select(
        funcionarios.c.cargo,
        func.avg(
            funcionarios.c.salario
        ).label("media_salarial")
    )
    .group_by(funcionarios.c.cargo)
)


with engine.connect() as conn:

    resultado = conn.execute(consulta)

    print("Média salarial por cargo:")

    for linha in resultado:
        print(linha)

Média salarial por cargo:
('Desenvolvedor Júnior', 3850.0000000000005)


**Nível 3: Avançado — ORM (Orientação a Objetos e Relacionamentos)**

In [16]:
Base = declarative_base()


class Departamento(Base):

    __tablename__ = "departamentos"

    id: Mapped[int] = mapped_column(
        Integer,
        primary_key=True,
        autoincrement=True
    )

    nome: Mapped[str] = mapped_column(
        String(100),
        nullable=False
    )

    funcionarios: Mapped[list["FuncionarioORM"]] = relationship(
        back_populates="departamento"
    )


class FuncionarioORM(Base):

    __tablename__ = "funcionarios_orm"

    id: Mapped[int] = mapped_column(
        Integer,
        primary_key=True,
        autoincrement=True
    )

    nome: Mapped[str] = mapped_column(
        String(100),
        nullable=False
    )

    cargo: Mapped[str] = mapped_column(
        String(100),
        nullable=False
    )

    salario: Mapped[float] = mapped_column(
        Float,
        nullable=False
    )

    departamento_id: Mapped[int] = mapped_column(
        ForeignKey("departamentos.id")
    )

    departamento: Mapped["Departamento"] = relationship(
        back_populates="funcionarios"
    )


In [17]:
Base.metadata.create_all(engine)

Session = sessionmaker(bind=engine)
sessao = Session()

departamento_ti = Departamento(
    nome="TI"
)

funcionario1 = FuncionarioORM(
    nome="Davi",
    cargo="Desenvolvedor Júnior",
    salario=3500.00
)

funcionario2 = FuncionarioORM(
    nome="João",
    cargo="Analista de Sistemas",
    salario=4500.00
)

departamento_ti.funcionarios.append(funcionario1)
departamento_ti.funcionarios.append(funcionario2)

sessao.add(departamento_ti)

sessao.commit()

In [18]:
consulta = (
    select(FuncionarioORM)
    .join(FuncionarioORM.departamento)
    .where(
        Departamento.nome == "TI"
    )
)


funcionarios_ti = sessao.execute(
    consulta
).scalars().all()


print("\nFuncionários do departamento de TI:")

for funcionario in funcionarios_ti:

    print(
        funcionario.nome,
        "-",
        funcionario.cargo,
        "-",
        funcionario.salario
    )

sessao.close()


Funcionários do departamento de TI:
Davi - Desenvolvedor Júnior - 3500.0
João - Analista de Sistemas - 4500.0
